# Homework 5: Deploying Machine Learning Models

Questions: [ML Zoomcamp 2026 – Homework 5](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/05-deployment/homework.md)

Model: the frozen 2026 lead-scoring release (`pipeline.bin`), a `DictVectorizer` + `LogisticRegression(solver="liblinear", C=1.0)` pipeline trained on [course_lead_scoring_2026.csv](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv).

Goal: load the model, serve it with FastAPI, and run the same prediction through the Docker container.

This module has its own pinned environment (Python 3.11.15, `uv.lock`). Run the notebook from inside `05-deployment`:

```bash
uv sync --locked
uv run --with jupyter jupyter lab homework.ipynb
```

In [1]:
import hashlib
import json
import pickle
import subprocess
import time
import tomllib
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

## Setup

Verify the artifact checksum before unpickling it. It must match the value in the homework and in `model_metadata.json`.

In [2]:
expected_sha = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
metadata = json.loads(Path("model_metadata.json").read_text())

model_sha = hashlib.sha256(Path("pipeline.bin").read_bytes()).hexdigest()
model_sha == expected_sha == metadata["model_sha256"]

True

## Q1. Environment check

Run `uv --version` and record the version. This is a setup check, not a graded answer.

In [3]:
!uv --version

uv 0.12.19 (bea138450 2026-09-24 aarch64-apple-darwin)


**Answer:** `uv 0.12.19` (the Dockerfile itself pins `uv 0.10.11`)

## Q2. Locked dependency

Which Scikit-Learn version is part of the 2026 reference environment?

- `1.6.1`
- `1.7.2`
- `1.8.0`
- `2.0.0`

In [4]:
lock = tomllib.loads(Path("uv.lock").read_text())
next(p["version"] for p in lock["package"] if p["name"] == "scikit-learn")

'1.7.2'

**Answer:** `1.7.2`

`pyproject.toml` pins `scikit-learn==1.7.2` and the lockfile resolves to the same version. It matters because the pickle has to be loaded with the same scikit-learn version that produced it.

## Q3. Load the model

Load `pipeline.bin` with `pickle` and report the probability of conversion for this lead, rounded to 3 decimals:

In [5]:
with open("pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)

print(pipeline)

Pipeline(steps=[('dictvectorizer', DictVectorizer()),
                ('logisticregression',
                 LogisticRegression(max_iter=1000, random_state=42,
                                    solver='liblinear'))])


In [6]:
lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

round(float(pipeline.predict_proba([lead])[0, 1]), 3)

0.533

**Answer:** 0.533

The pipeline bundles the `DictVectorizer` with the model, so a raw dict goes straight into `predict_proba`. Every field is present, so the API's missing-value rules (`NA` / training medians) don't kick in.

## Q4. Serve the model

Start the provided FastAPI app (`predict.py`) with uvicorn on port 9696 and send this second lead to `POST /predict`. Report `conversion_probability` rounded to 3 decimals.

In [7]:
URL = "http://localhost:9696"

client = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}


def get_health():
    with urlopen(f"{URL}/health") as response:
        return json.load(response)


def wait_for_health(timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            return get_health()
        except (URLError, ConnectionError):
            time.sleep(0.5)
    raise TimeoutError(f"API at {URL} did not become healthy")


def predict(lead):
    request = Request(
        f"{URL}/predict",
        data=json.dumps(lead).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urlopen(request) as response:
        return json.load(response)

In [8]:
server = subprocess.Popen(
    ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
try:
    wait_for_health()
    result = predict(client)
finally:
    server.terminate()
    server.wait()

result

{'conversion_probability': 0.769799, 'conversion': True}

In [9]:
round(result["conversion_probability"], 3)

0.77

**Answer:** 0.770

The response holds a probability in `[0, 1]` and a boolean `conversion` (`True`, since 0.77 ≥ 0.5).

## Q5. Container configuration

Which Python base-image tag is declared in the Dockerfile?

- `python:3.9-slim-bullseye`
- `python:3.11.15-slim-bookworm`
- `python:3.12-slim-bookworm`
- `python:3.13.10-slim-bookworm`

In [10]:
!cat Dockerfile

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /code
COPY pyproject.toml uv.lock .python-version ./
RUN uv sync --locked --no-dev

COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./

EXPOSE 9696
ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]


**Answer:** `python:3.11.15-slim-bookworm`

Both the base image and the `uv` image are pinned by digest as well as tag, and dependencies are installed with `uv sync --locked`, so the build is reproducible.

## Q6. Run the container

Build and run the image, send the Q4 request to it, and check that `/health` returns `status: ok` and the model checksum from `model_metadata.json`.

```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

In [11]:
!docker build -q -t zoomcamp-model:2026-hw5 .
!docker run -d --rm --name hw5 -p 9696:9696 zoomcamp-model:2026-hw5

sha256:ca4f25817471230a0b0029e1cdb590ddacacfb7973639cfcd6fc1a7bdba3fec0


9e383d870eec4ec9807b2d09892039605052b976a4809ada558377f3d41a940e


In [12]:
try:
    health = wait_for_health()
    result = predict(client)
finally:
    subprocess.run(["docker", "stop", "hw5"], check=True, stdout=subprocess.DEVNULL)

health

{'status': 'ok',
 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1',
 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}

In [13]:
health["status"] == "ok" and health["model_sha256"] == metadata["model_sha256"]

True

In [14]:
result, round(result["conversion_probability"], 3)

({'conversion_probability': 0.769799, 'conversion': True}, 0.77)

**Answer:** 0.770

Same as Q4. The container serves the same frozen artifact with the same locked dependencies, so the prediction matches exactly (0.769799).

## Summary

| Question | Answer |
|---|---|
| Q1. `uv` version (not graded) | `uv 0.12.19` |
| Q2. Locked scikit-learn version | `1.7.2` |
| Q3. Probability via pickle | 0.533 |
| Q4. Probability via local API | 0.770 |
| Q5. Dockerfile base image | `python:3.11.15-slim-bookworm` |
| Q6. Probability via container | 0.770 |